# Speech Emotion Recognition using Traditional Machine Learning

This notebook implements Speech Emotion Recognition (SER) using **traditional machine learning** approaches only:
- Random Forest
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)
- Logistic Regression

Features extracted: MFCC, ZCR, Chroma, RMS, Mel Spectrogram

In [ ]:
# Install required packages
!pip install pandas numpy librosa matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Audio processing
import librosa
import librosa.display

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from IPython.display import Audio

import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 1. Dataset Setup

Define paths to emotion datasets: RAVDESS, CREMA-D, TESS, SAVEE

In [ ]:
# Dataset paths
Ravdess = "dataset/ravdess-emotional-speech-audio/audio_speech_actors_01-24/"
Crema = "dataset/cremad/AudioWAV/"
Tess = "dataset/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data/TESS Toronto emotional speech set data/"
Savee = "dataset/surrey-audiovisual-expressed-emotion-savee/ALL/"

In [ ]:
# Load RAVDESS dataset
ravdess_directory_list = os.listdir(Ravdess)
file_emotion = []
file_path = []

for dir in ravdess_directory_list:
    actor = os.listdir(Ravdess + dir)
    for file in actor:
        part = file.split('.')[0].split('-')
        file_emotion.append(int(part[2]))
        file_path.append(Ravdess + dir + '/' + file)

emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Ravdess_df = pd.concat([emotion_df, path_df], axis=1)

Ravdess_df.Emotions.replace({
    1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
    5: 'angry', 6: 'fear', 7: 'disgust', 8: 'surprise'
}, inplace=True)

Ravdess_df.head()

In [ ]:
# Load CREMA-D dataset
crema_directory_list = os.listdir(Crema)
file_emotion = []
file_path = []

for file in crema_directory_list:
    file_path.append(Crema + file)
    part = file.split('_')
    if part[2] == 'SAD':
        file_emotion.append('sad')
    elif part[2] == 'ANG':
        file_emotion.append('angry')
    elif part[2] == 'DIS':
        file_emotion.append('disgust')
    elif part[2] == 'FEA':
        file_emotion.append('fear')
    elif part[2] == 'HAP':
        file_emotion.append('happy')
    elif part[2] == 'NEU':
        file_emotion.append('neutral')
    else:
        file_emotion.append('Unknown')

emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Crema_df = pd.concat([emotion_df, path_df], axis=1)
Crema_df.head()

In [ ]:
# Load TESS dataset
tess_directory_list = os.listdir(Tess)
file_emotion = []
file_path = []

for dir in tess_directory_list:
    directories = os.listdir(Tess + dir)
    for file in directories:
        part = file.split('.')[0].split('_')[2]
        if part == 'ps':
            file_emotion.append('surprise')
        else:
            file_emotion.append(part)
        file_path.append(Tess + dir + '/' + file)

emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Tess_df = pd.concat([emotion_df, path_df], axis=1)
Tess_df.head()

In [ ]:
# Load SAVEE dataset
savee_directory_list = os.listdir(Savee)
file_emotion = []
file_path = []

for file in savee_directory_list:
    file_path.append(Savee + file)
    part = file.split('_')[1]
    ele = part[:-6]
    if ele == 'a':
        file_emotion.append('angry')
    elif ele == 'd':
        file_emotion.append('disgust')
    elif ele == 'f':
        file_emotion.append('fear')
    elif ele == 'h':
        file_emotion.append('happy')
    elif ele == 'n':
        file_emotion.append('neutral')
    elif ele == 'sa':
        file_emotion.append('sad')
    else:
        file_emotion.append('surprise')

emotion_df = pd.DataFrame(file_emotion, columns=['Emotions'])
path_df = pd.DataFrame(file_path, columns=['Path'])
Savee_df = pd.concat([emotion_df, path_df], axis=1)
Savee_df.head()

In [ ]:
# Combine all datasets
data_path = pd.concat([Ravdess_df, Crema_df, Tess_df, Savee_df], axis=0)
data_path.to_csv("data_path.csv", index=False)
data_path.head()

## 2. Emotion Distribution Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.title('Count of Emotions', size=16, weight='bold')
sns.countplot(data=data_path, x='Emotions', palette='Set2', edgecolor='black')
plt.ylabel('Count', size=12, weight='bold')
plt.xlabel('Emotions', size=12, weight='bold')
plt.xticks(rotation=45, ha='right', size=10)
sns.despine(top=True, right=True)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Waveform and Spectrogram Visualization

In [ ]:
def create_waveplot(data, sr, e):
    plt.figure(figsize=(10, 3))
    plt.title('Waveplot for audio with {} emotion'.format(e), size=15)
    librosa.display.waveshow(y=data, sr=sr)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.show()

def create_spectrogram(data, sr, e):
    X = librosa.stft(data)
    Xdb = librosa.amplitude_to_db(abs(X))
    plt.figure(figsize=(12, 3))
    plt.title('Spectrogram for audio with {} emotion'.format(e), size=15)
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.show()

In [ ]:
# Visualize 'sad' emotion
emotion = 'sad'
path = np.array(data_path.Path[data_path.Emotions == emotion])[1]
data, sampling_rate = librosa.load(path)
create_waveplot(data, sampling_rate, emotion)
create_spectrogram(data, sampling_rate, emotion)
Audio(path)

## 4. Feature Extraction (MFCC + ZCR + Chroma + RMS + Mel)

In [ ]:
def extract_features(data, sample_rate):
    """Extract audio features: ZCR, Chroma, MFCC, RMS, Mel Spectrogram"""
    result = np.array([])
    
    # Zero Crossing Rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y=data).T, axis=0)
    result = np.hstack((result, zcr))
    
    # Chroma STFT
    stft = np.abs(librosa.stft(data))
    chroma_stft = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
    result = np.hstack((result, chroma_stft))
    
    # MFCC (20 coefficients)
    mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sample_rate, n_mfcc=20).T, axis=0)
    result = np.hstack((result, mfcc))
    
    # RMS
    rms = np.mean(librosa.feature.rms(y=data).T, axis=0)
    result = np.hstack((result, rms))
    
    # Mel Spectrogram
    mel = np.mean(librosa.feature.melspectrogram(y=data, sr=sample_rate).T, axis=0)
    result = np.hstack((result, mel))
    
    return result

def get_features(path):
    """Load audio and extract features with simple augmentation"""
    # Load audio with 2.5 sec duration, 0.6 sec offset
    data, sample_rate = librosa.load(path, duration=2.5, offset=0.6)
    
    # Extract features from original
    res1 = extract_features(data, sample_rate)
    result = np.array(res1)
    
    # Add noise augmentation
    noise_amp = 0.035 * np.random.uniform() * np.amax(data)
    noise_data = data + noise_amp * np.random.normal(size=data.shape[0])
    res2 = extract_features(noise_data, sample_rate)
    result = np.vstack((result, res2))
    
    # Time stretch augmentation
    stretched = librosa.effects.time_stretch(data, rate=0.8)
    res3 = extract_features(stretched, sample_rate)
    result = np.vstack((result, res3))
    
    return result

In [ ]:
# Extract features from all audio files
X, Y = [], []

for path, emotion in zip(data_path.Path, data_path.Emotions):
    feature = get_features(path)
    for ele in feature:
        X.append(ele)
        Y.append(emotion)

print(f"Total samples: {len(X)}")
print(f"Feature dimension: {len(X[0])}")

In [ ]:
# Create DataFrame and save
Features = pd.DataFrame(X)
Features['labels'] = Y
Features.to_csv('features.csv', index=False)
Features.head()

## 5. Data Preparation

In [ ]:
# Prepare features and labels
X = Features.iloc[:, :-1].values
Y = Features['labels'].values

# Encode labels
label_encoder = LabelEncoder()
Y_encoded = label_encoder.fit_transform(Y)

print(f"Classes: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")

In [ ]:
# Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    X, Y_encoded, test_size=0.25, random_state=0, shuffle=True, stratify=Y_encoded
)

print(f"Training samples: {x_train.shape[0]}")
print(f"Testing samples: {x_test.shape[0]}")

In [ ]:
# Scale features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(f"Scaled training data shape: {x_train.shape}")
print(f"Scaled testing data shape: {x_test.shape}")

## 6. Traditional Machine Learning Models

In [ ]:
# Define models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, max_depth=20),
    'SVM': SVC(random_state=42, kernel='rbf', C=1.0, gamma='scale'),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, C=1.0)
}

results = {}

In [ ]:
# Train and evaluate each model
print("="*60)
print("TRADITIONAL MACHINE LEARNING MODEL COMPARISON")
print("="*60)

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    # Train
    model.fit(x_train, y_train)
    
    # Predict
    y_pred = model.predict(x_test)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"\n{name} Accuracy: {accuracy*100:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## 7. Results Comparison

In [ ]:
# Display results summary
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)

results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)
results_df['Accuracy (%)'] = results_df['Accuracy'] * 100

print(results_df.to_string(index=False))

best_model = results_df.iloc[0]['Model']
best_accuracy = results_df.iloc[0]['Accuracy (%)']
print(f"\n🏆 Best Model: {best_model} with {best_accuracy:.2f}% accuracy")

In [ ]:
# Plot accuracy comparison
plt.figure(figsize=(10, 6))
plt.barh(results_df['Model'], results_df['Accuracy (%)'], color='skyblue', edgecolor='black')
plt.xlabel('Accuracy (%)', size=12, weight='bold')
plt.ylabel('Model', size=12, weight='bold')
plt.title('Traditional ML Model Comparison', size=14, weight='bold')
plt.xlim([0, 100])
for i, v in enumerate(results_df['Accuracy (%)']):
    plt.text(v + 1, i, f'{v:.2f}%', va='center', fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 8. Confusion Matrix for Best Model

In [ ]:
# Get best model and generate predictions
best_model_name = results_df.iloc[0]['Model']
best_model_obj = models[best_model_name]
y_pred_best = best_model_obj.predict(x_test)

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Normalized Count'})
plt.title(f'Confusion Matrix - {best_model_name}', size=16, weight='bold')
plt.xlabel('Predicted Labels', size=12, weight='bold')
plt.ylabel('Actual Labels', size=12, weight='bold')
plt.tight_layout()
plt.show()

## 9. Save Best Model

In [ ]:
import joblib

# Save the best model
joblib.dump(best_model_obj, f'{best_model_name.replace(" ", "_").lower()}_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

print(f"✅ Saved {best_model_name} model to '{best_model_name.replace(' ', '_').lower()}_model.pkl'")
print(f"✅ Saved scaler to 'scaler.pkl'")
print(f"✅ Saved label encoder to 'label_encoder.pkl'")